<a href="https://colab.research.google.com/github/cabareno/Marketing_exercise/blob/main/Ejercicio_empresa_marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Contexto del Negocio

El equipo de Paid Media & Growth ha ejecutado una campaña internacional de performance omnicanal. Al descargar el reporte consolidado crudo desde el agregador de pauta publicitaria, el equipo de analítica detectó que el archivo presenta truncamientos de filas, registros de mercados ajenos al foco local, formatos de fecha inestables, IDs corruptos y nulos críticos en la columna de conversiones y estados de campaña. Como Data Analytics Engineer, tu objetivo es diseñar un pipeline de limpieza, aplicar las reglas de gobernanza del negocio y estructurar un modelo analítico óptimo para ser consumido en Snowflake y Looker Studio para calcular el ROAS y la efectividad de la pauta.

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Librerias

In [ ]:
# Tratamiento de datos y operaciones
import pandas as pd
import numpy as np

# Conexion de sesion pyspark
from pyspark.sql import SparkSession
from pyspark import SparkContext
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = SparkContext.getOrCreate()

from pyspark.sql import SQLContext
sqlCtx = SQLContext(sc)

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


## Lectura de bases

In [ ]:
#Esta ruta debera ser ajustada acorde a la carpeta donde quede como compartida
path0 = '/content/drive/MyDrive/prueba_Asylum_Marketing/reporte_performance_crudo (1) (1).csv'
data = pd.read_csv(path0)
data.head(5)

,Unnamed: 0,Campaign_ID,Fecha_Métrica,Canal_Publicitario,Landing_Page,Presupuesto_Invertido,Clics,Leads_Captados,Unnamed: 4,Ventas_Totales,Estado_Aprobación
0,0,WMX500000,7/04/2026 4:38 am,Display_Ads,global-shop.com/it,300.58,12863,1316,-0.524520,1267.70,True
1,1,WMX500001,11/04/2026 8:30 am,YouTube Video,cyber-ofertas.com.mx/home,1631.85,2100,107,0.489375,4450.14,NaN
2,2,WMX500002,26/04/2026 3:32 am,Facebook Ads,global-shop.com/it,1650.25,11048,329,-1.222128,4598.34,True
3,3,WMX500003,11/04/2026 8:45 pm,paid_social,cyber-ofertas.com.mx/home,0.00,9960,366,0.712998,0.00,True
4,4,WMX500004,24/04/2026 3:31 am,Google Ads,global-shop.com/it,1649.83,6089,223,-0.240325,5903.05,True


In [ ]:
# Se hace una validacion inicial de la composicion de la base tanto en columnas como en formatos
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             100 non-null    int64  
 1   Campaign_ID            100 non-null    object 
 2   Fecha_Métrica          99 non-null     object 
 3   Canal_Publicitario     100 non-null    object 
 4   Landing_Page           100 non-null    object 
 5   Presupuesto_Invertido  100 non-null    float64
 6   Clics                  100 non-null    int64  
 7   Leads_Captados         100 non-null    int64  
 8   Unnamed: 4             100 non-null    float64
 9   Ventas_Totales         100 non-null    float64
 10  Estado_Aprobación      81 non-null     object 
dtypes: float64(3), int64(3), object(5)
memory usage: 8.7+ KB


## 1. Saneamiento Estructural e Ingesta Cruda

In [ ]:
# Se removeran las columnas que no trae un encabezado consideradas fantasma.
colum_ori = data.columns.to_list() #pasamos las columnas iniciales a tipo lista
print(colum_ori)

['Unnamed: 0', 'Campaign_ID', 'Fecha_Métrica', 'Canal_Publicitario', 'Landing_Page', 'Presupuesto_Invertido', 'Clics', 'Leads_Captados', 'Unnamed: 4', 'Ventas_Totales', 'Estado_Aprobación']


In [ ]:
#A partir del data inicial y la seleccion de columnas se crea un nuevo dataframe de trabajo
delete_items = ['Unnamed: 4','Unnamed: 0'] # se seleccionan los items de la lista que no se desean
data2 = data.drop(columns=delete_items)
data2.head(3)

,Campaign_ID,Fecha_Métrica,Canal_Publicitario,Landing_Page,Presupuesto_Invertido,Clics,Leads_Captados,Ventas_Totales,Estado_Aprobación
0,WMX500000,7/04/2026 4:38 am,Display_Ads,global-shop.com/it,300.58,12863,1316,1267.70,True
1,WMX500001,11/04/2026 8:30 am,YouTube Video,cyber-ofertas.com.mx/home,1631.85,2100,107,4450.14,NaN
2,WMX500002,26/04/2026 3:32 am,Facebook Ads,global-shop.com/it,1650.25,11048,329,4598.34,True


In [ ]:
# Se realizara una limpieza a la columna Campain_ID para asegurar consistencia en el formato
# Lo primero que se hara es ver como se compone la columna, que valores inconsistentes trae
print(data2['Campaign_ID'].value_counts())

Campaign_ID
9.87E+06     10
WMX500001     1
WMX500000     1
WMX500003     1
WMX500004     1
             ..
WPT700015     1
WPT700016     1
WPT700017     1
WPT700018     1
WPT700019     1
Name: count, Length: 91, dtype: int64


In [ ]:
# Se observa que la inconsistencia de informacion en la columna se da por valores uqe viene como numeros que rompen el esquema del formato string.
# Como no se conoce exactamente la estructura para asignar el nombre de camapaña se dejaran como valores "UNKNOW" para analisis posteriores.
data2['Campaign_ID'] = np.where(data2['Campaign_ID'].str.match(r'^[0-9]+(\.[0-9]+)?([eE][+-]?[0-9]+)?$', na=False),'UNKNOWN',
                                data2['Campaign_ID'].str.strip()) # Esta ultima parte elimina posibles espacios en los nombres
print(data2['Campaign_ID'].value_counts())

Campaign_ID
UNKNOWN      10
WMX500001     1
WMX500000     1
WMX500003     1
WMX500004     1
             ..
WPT700015     1
WPT700016     1
WPT700017     1
WPT700018     1
WPT700019     1
Name: count, Length: 91, dtype: int64


In [ ]:
print(data2['Campaign_ID'].to_list()) #Se imprime la lista de los valores que toma la columna para validar que no tenga espacios

['WMX500000', 'WMX500001', 'WMX500002', 'WMX500003', 'WMX500004', 'WMX500005', 'WMX500006', 'WMX500007', 'WMX500008', 'WMX500009', 'WMX500010', 'WMX500011', 'WMX500012', 'WMX500013', 'WMX500014', 'WMX500015', 'WMX500016', 'WMX500017', 'WMX500018', 'WMX500019', 'WMX500020', 'WMX500021', 'WMX500022', 'WMX500023', 'WMX500024', 'WMX500025', 'WMX500026', 'WMX500027', 'WMX500028', 'WMX500029', 'WMX500030', 'WMX500031', 'WMX500032', 'WMX500033', 'WMX500034', 'WMX500035', 'WMX500036', 'WMX500037', 'WMX500038', 'WMX500039', 'WCO600000', 'WCO600001', 'WCO600002', 'WCO600003', 'WCO600004', 'WCO600005', 'WCO600006', 'WCO600007', 'WCO600008', 'WCO600009', 'WCO600010', 'WCO600011', 'WCO600012', 'WCO600013', 'WCO600014', 'WCO600015', 'WCO600016', 'WCO600017', 'WCO600018', 'WCO600019', 'WCO600020', 'WCO600021', 'WCO600022', 'WCO600023', 'WCO600024', 'WCO600025', 'WCO600026', 'WCO600027', 'WCO600028', 'WCO600029', 'UNKNOWN', 'UNKNOWN', 'UNKNOWN', 'UNKNOWN', 'UNKNOWN', 'UNKNOWN', 'UNKNOWN', 'UNKNOWN', '

In [ ]:
# Finalmente la columna sera casteada para que todos los valores sean de tipo string
data2['Campaign_ID'] = data2['Campaign_ID'].astype('str')
print(type(data2['Campaign_ID'][0])) #Se evalua que el casteado sea exitoso imprimiendo el tipo de valor del primer elemento de la columna

<class 'str'>


In [ ]:
#Tarea 1.3. Pasaremos los valores de la columna  Fecha_Métrica  a valores de tipo fecha acorde con la norma ISO 8601
print(data2['Fecha_Métrica'].value_counts()) # Se valida como viene compuesta inicialmente la columna

Fecha_Métrica
7/04/2026 4:38 am     1
11/04/2026 8:30 am    1
26/04/2026 3:32 am    1
11/04/2026 8:45 pm    1
24/04/2026 3:31 am    1
                     ..
27/04/2026 2:30 pm    1
1/04/2026 1:14 pm     1
16/04/2026 3:13 am    1
17/04/2026 1:53 pm    1
14/04/2026 5:15 am    1
Name: count, Length: 99, dtype: int64


In [ ]:
data3 = data2.copy() # Primero haremos una copia de la data
data3['Fecha_Métrica'] = pd.to_datetime(data3.Fecha_Métrica, format='mixed') #Se pasa a tipo fecha y como no estamos seguros del formato se pasa a por "mixed"
print(data3['Fecha_Métrica'].value_counts())

Fecha_Métrica
2026-07-04 04:38:00    1
2026-11-04 08:30:00    1
2026-04-26 03:32:00    1
2026-11-04 20:45:00    1
2026-04-24 03:31:00    1
                      ..
2026-04-27 14:30:00    1
2026-01-04 13:14:00    1
2026-04-16 03:13:00    1
2026-04-17 13:53:00    1
2026-04-14 05:15:00    1
Name: count, Length: 99, dtype: int64


In [ ]:
# Aunque al convertirla a tipo fecha con formato mixto parece cumplir con la estructura esperada, se asegura explícitamente que la columna quede en el formato deseado
data3['Fecha_Métrica'] = data3['Fecha_Métrica'].dt.strftime('%Y-%m-%d %H:%M:%S')
print(data3['Fecha_Métrica'].value_counts())

Fecha_Métrica
2026-07-04 04:38:00    1
2026-11-04 08:30:00    1
2026-04-26 03:32:00    1
2026-11-04 20:45:00    1
2026-04-24 03:31:00    1
                      ..
2026-04-27 14:30:00    1
2026-01-04 13:14:00    1
2026-04-16 03:13:00    1
2026-04-17 13:53:00    1
2026-04-14 05:15:00    1
Name: count, Length: 99, dtype: int64


## 2. Reglas de Negocio, Gobierno de Datos y Mapping Engine

In [ ]:
# Se valida como esta compuesta la variable para saber como agruparla
print(data3['Canal_Publicitario'].value_counts())

Canal_Publicitario
Programmatic       13
Display_Ads        11
TikTok Paid        10
paid_social         9
Instagram_Pauta     9
Google Ads          9
Paid Search         9
google_search       8
AdWords             7
YouTube Video       6
Facebook Ads        6
DV360               3
Name: count, dtype: int64


In [ ]:
data3[~data3['Canal_Publicitario'].str.contains('Google Ads|Paid Search|google_search|AdWords|DV360')]

,Campaign_ID,Fecha_Métrica,Canal_Publicitario,Landing_Page,Presupuesto_Invertido,Clics,Leads_Captados,Ventas_Totales,Estado_Aprobación
0,WMX500000,2026-07-04 04:38:00,Display_Ads,global-shop.com/it,300.58,12863,1316,1267.70,True
1,WMX500001,2026-11-04 08:30:00,YouTube Video,cyber-ofertas.com.mx/home,1631.85,2100,107,4450.14,NaN
2,WMX500002,2026-04-26 03:32:00,Facebook Ads,global-shop.com/it,1650.25,11048,329,4598.34,True
3,WMX500003,2026-11-04 20:45:00,paid_social,cyber-ofertas.com.mx/home,0.00,9960,366,0.00,True
5,WMX500005,2026-02-04 20:53:00,Facebook Ads,promo.co/landing,1320.40,1560,62,1648.09,NaN
...,...,...,...,...,...,...,...,...,...
92,WPT700012,2026-12-04 11:31:00,Programmatic,blackfriday.de/shop,693.99,11323,391,983.39,False
95,WPT700015,2026-04-27 14:30:00,paid_social,blackfriday.de/shop,840.14,8041,1065,2574.69,True
97,WPT700017,2026-04-16 03:13:00,Display_Ads,global-shop.com/it,1494.90,5669,303,3992.73,False
98,WPT700018,2026-04-17 13:53:00,Programmatic,promo.co/landing,1546.03,4400,185,2330.17,True


In [ ]:
# Creacion de funcion
# Se crea una funcion que agrupe los valores originales de la columna segun sea cada caso
def limpia_canal(base,columna):
  condi = [base[columna].str.contains('Google Ads|Paid Search|google_search|AdWords|DV360'),
           base[columna].str.contains('TikTok|paid_social|Instagram|Facebook'),
           base[columna].str.contains('Programmatic|Display_Ads|YouTube')]
  valores = ['PAID SEARCH','PAID SOCIAL','PROGRAMMATIC & VIDEO']
  base[f'{columna}_new'] = np.select(condi, valores, default=None)
  return base[f'{columna}_new']

In [ ]:
data4 = data3.copy()# Se crea una copia por seguridad
# Posterior a esto se aplica la funcion previamente creada y se valida
limpia_canal(data4, 'Canal_Publicitario')
print(data4['Canal_Publicitario_new'].value_counts())

Canal_Publicitario_new
PAID SEARCH             36
PAID SOCIAL             34
PROGRAMMATIC & VIDEO    30
Name: count, dtype: int64


In [ ]:
# Se reeemplaza la columna creada por la que debe ser la resultante de la base
data4['Canal_Publicitario'] = data4['Canal_Publicitario_new']
data4 = data4.drop(columns=['Canal_Publicitario_new'])

In [ ]:
data4.head(2)

,Campaign_ID,Fecha_Métrica,Canal_Publicitario,Landing_Page,Presupuesto_Invertido,Clics,Leads_Captados,Ventas_Totales,Estado_Aprobación
0,WMX500000,2026-07-04 04:38:00,PROGRAMMATIC & VIDEO,global-shop.com/it,300.58,12863,1316,1267.70,True
1,WMX500001,2026-11-04 08:30:00,PROGRAMMATIC & VIDEO,cyber-ofertas.com.mx/home,1631.85,2100,107,4450.14,NaN


In [ ]:
# Validacion canal_publicitario
data4['Canal_Publicitario'].value_counts()

,count
Canal_Publicitario,
PAID SEARCH,36
PAID SOCIAL,34
PROGRAMMATIC & VIDEO,30


In [ ]:
#Se ajusta la base para completar con False los casos nulos que puedan afectar futuros calculos
data4['Ventas_Totales'] = data4['Ventas_Totales'].fillna(0)
data4['Leads_Captados'] = data4['Leads_Captados'].fillna(0)
data4['Clics'] = data4['Clics'].fillna(0)
data4['Presupuesto_Invertido'] = data4['Presupuesto_Invertido'].fillna(0)
data4['Estado_Aprobación'] = data4['Estado_Aprobación'].fillna(False)

/tmp/ipykernel_1069/757571328.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data4['Estado_Aprobación'] = data4['Estado_Aprobación'].fillna(False)


In [ ]:
# Esta funcion realiza el calculo de las roas preveendo casos donde el presupuesto invertido sea cero o nulo
def roas(numerador, divisor, base):
  return np.where(base[divisor] >0, base[numerador]/base[divisor], 0)
# Esta funcion clasifica en True o False el objetivo de la campaña
def taget_campaing(fecha, estado, campaña, landing,base):
  return np.where((base[fecha].notnull()) & (base[estado] == True) & (base[campaña].str.startswith('WMX') | base[landing].str.contains('.mx')),
                  True, False)

In [ ]:
# Se aplican las dos funciones creadas previamente
data4['ROAS'] = roas('Ventas_Totales','Presupuesto_Invertido', data4)
data4['Is_Target_Campaign'] = taget_campaing('Fecha_Métrica','Estado_Aprobación','Campaign_ID','Landing_Page',data4)

In [ ]:
#Validacion de Is_Target_Campaign
data4['Is_Target_Campaign'].value_counts()

,count
Is_Target_Campaign,
False,67
True,33


## 3. Modelado de Capas dbt / Snowflake (Pensamiento Analítico)

La solicitud implica trabajar con lenguaje SQL. Por ello, en el entorno de Colab y para aplicar los requerimientos, se lee la base desde su origen como una tabla que posteriormente pueda ser contextualizada en un ambiente SQL.

In [ ]:
data_1 = sqlCtx.read.option("header",True).csv("/content/drive/MyDrive/prueba_Asylum_Marketing/reporte_performance_crudo (1) (1).csv")
data_1.show(3)

+----------+-----------+------------------+------------------+--------------------+---------------------+-----+--------------+-------------------+--------------+-----------------+
|Unnamed: 0|Campaign_ID|     Fecha_Métrica|Canal_Publicitario|        Landing_Page|Presupuesto_Invertido|Clics|Leads_Captados|         Unnamed: 4|Ventas_Totales|Estado_Aprobación|
+----------+-----------+------------------+------------------+--------------------+---------------------+-----+--------------+-------------------+--------------+-----------------+
|         0|  WMX500000| 7/04/2026 4:38 am|       Display_Ads|  global-shop.com/it|               300.58|12863|          1316|-0.5245202662797737|        1267.7|             true|
|         1|  WMX500001|11/04/2026 8:30 am|     YouTube Video|cyber-ofertas.com...|              1631.85| 2100|           107|0.48937456122791806|       4450.14|             NULL|
|         2|  WMX500002|26/04/2026 3:32 am|      Facebook Ads|  global-shop.com/it|              165

In [ ]:
data_1.printSchema()

root
 |-- Unnamed: 0: string (nullable = true)
 |-- Campaign_ID: string (nullable = true)
 |-- Fecha_Métrica: string (nullable = true)
 |-- Canal_Publicitario: string (nullable = true)
 |-- Landing_Page: string (nullable = true)
 |-- Presupuesto_Invertido: string (nullable = true)
 |-- Clics: string (nullable = true)
 |-- Leads_Captados: string (nullable = true)
 |-- Unnamed: 4: string (nullable = true)
 |-- Ventas_Totales: string (nullable = true)
 |-- Estado_Aprobación: string (nullable = true)



In [ ]:
#La base leida se reginstra como una temporal llamada Base para ppoderla manipular con lenguaje SQL
data_1.registerTempTable("base")

/usr/local/lib/python3.12/dist-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [ ]:
# Tarea 3.1. La query creada a continuacion es un CTE que ayuda en la limpieza de la base antes de realizar calculos.
# Permite eliminar nulos, espacion, ajustar formatos y tranformar caracteres extraños
data_limpia = sqlCtx.sql("""
with base_limpia as (
select Campaign_ID,`Fecha_Métrica`,Canal_Publicitario,Landing_Page,Presupuesto_Invertido,Clics,Leads_Captados,Ventas_Totales,`Estado_Aprobación`
from base),
replace_column as (
Select case when Campaign_ID rlike '^[0-9]+(\.[0-9]+)?([eE][+-]?[0-9]+)?$' then 'unknown' else trim(Campaign_ID) end as Campaign_ID,
`Fecha_Métrica`,Canal_Publicitario,Landing_Page,Presupuesto_Invertido,Clics,Leads_Captados,Ventas_Totales,`Estado_Aprobación`
from  base_limpia
),
nulos_estado as
( Select Campaign_ID,`Fecha_Métrica`,Canal_Publicitario,Landing_Page,
COALESCE(cast(Presupuesto_Invertido as float),0) as Presupuesto_Invertido,
COALESCE(Clics,0) as Clics,
COALESCE(Leads_Captados,0) as Leads_Captados,
COALESCE(cast(Ventas_Totales as float),0) as Ventas_Totales,
COALESCE(`Estado_Aprobación`, False) as `Estado_Aprobación`
from replace_column
),
fechas as (
Select Campaign_ID,
date_format(try_to_timestamp(`Fecha_Métrica`, 'd/MM/yyyy h:mm a'),'yyyy-MM-dd HH:mm:ss') as `Fecha_Métrica`,
Canal_Publicitario,Landing_Page,Presupuesto_Invertido,Clics,Leads_Captados,Ventas_Totales,`Estado_Aprobación`
from nulos_estado)
select * from fechas
""")
data_limpia.show(5)

<>:8: SyntaxWarning: invalid escape sequence '\.'
<>:8: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_1069/3379528560.py:8: SyntaxWarning: invalid escape sequence '\.'
  Select case when Campaign_ID rlike '^[0-9]+(\.[0-9]+)?([eE][+-]?[0-9]+)?$' then 'unknown' else trim(Campaign_ID) end as Campaign_ID,


+-----------+-------------------+------------------+--------------------+---------------------+-----+--------------+-----------------+-----------------+
|Campaign_ID|      Fecha_Métrica|Canal_Publicitario|        Landing_Page|Presupuesto_Invertido|Clics|Leads_Captados|   Ventas_Totales|Estado_Aprobación|
+-----------+-------------------+------------------+--------------------+---------------------+-----+--------------+-----------------+-----------------+
|  WMX500000|2026-04-07 04:38:00|       Display_Ads|  global-shop.com/it|    300.5799865722656|12863|          1316|1267.699951171875|             true|
|  WMX500001|2026-04-11 08:30:00|     YouTube Video|cyber-ofertas.com...|   1631.8499755859375| 2100|           107| 4450.14013671875|            false|
|  WMX500002|2026-04-26 03:32:00|      Facebook Ads|  global-shop.com/it|              1650.25|11048|           329|    4598.33984375|             true|
|  WMX500003|2026-04-11 20:45:00|       paid_social|cyber-ofertas.com...|         

In [ ]:
# Al validar con una view que todo esta ok, se regitra nuevamente la base resultante para continuar
data_limpia.registerTempTable("base2")

In [ ]:
# En esta query finalmente realizamos los calculos solcitados, asi como el ajuste de columna en agrupaciones mas claras
# para el consumidor final
data_transforma = sqlCtx.sql("""
with estandar as (
Select Campaign_ID, `Fecha_Métrica`,
case when REGEXP_LIKE(Canal_Publicitario, 'Google Ads|Paid Search|google_search|AdWords|DV360') then 'PAID SEARCH'
      when REGEXP_LIKE(Canal_Publicitario, 'TikTok|paid_social|Instagram|Facebook') then 'PAID SOCIAL'
      when REGEXP_LIKE(Canal_Publicitario, 'Programmatic|Display_Ads|YouTube') then 'PROGRAMMATIC & VIDEO'
      else Canal_Publicitario end as Canal_Publicitario,
Landing_Page,Presupuesto_Invertido,Clics,Leads_Captados,Ventas_Totales,`Estado_Aprobación`
from base2),
fct_performance_marketing_mx  as (
Select  *,
        case when Presupuesto_Invertido = 0 then 0 else Ventas_Totales/Presupuesto_Invertido end as ROAS,
        case when `Fecha_Métrica` is not null  and `Estado_Aprobación` = True and (Campaign_ID like 'WMX%' or Landing_Page like '%.mx%' )
              then TRUE else False end as Is_Target_Campaign
from estandar
)
Select * from fct_performance_marketing_mx
""")
data_transforma.show(5)

+-----------+-------------------+--------------------+--------------------+---------------------+-----+--------------+-----------------+-----------------+------------------+------------------+
|Campaign_ID|      Fecha_Métrica|  Canal_Publicitario|        Landing_Page|Presupuesto_Invertido|Clics|Leads_Captados|   Ventas_Totales|Estado_Aprobación|              ROAS|Is_Target_Campaign|
+-----------+-------------------+--------------------+--------------------+---------------------+-----+--------------+-----------------+-----------------+------------------+------------------+
|  WMX500000|2026-04-07 04:38:00|PROGRAMMATIC & VIDEO|  global-shop.com/it|    300.5799865722656|12863|          1316|1267.699951171875|             true| 4.217512834531629|              true|
|  WMX500001|2026-04-11 08:30:00|PROGRAMMATIC & VIDEO|cyber-ofertas.com...|   1631.8499755859375| 2100|           107| 4450.14013671875|            false|2.7270522433417126|             false|
|  WMX500002|2026-04-26 03:32:00|  

In [ ]:
#Validacion Is_Target_Campaign
data_transforma.groupby('Is_Target_Campaign').count().show()

+------------------+-----+
|Is_Target_Campaign|count|
+------------------+-----+
|              true|   33|
|             false|   67|
+------------------+-----+



Se concluye que el ejercicio esta ok al coincidir los valores de la variable
`Is_Target_Campaign` usando Python y usando SQL

